数据排序与聚合是数据分析中非常常见且重要的操作，特别是在 **大数据集** 中的数据分析时。

**排序** 帮助我们按特定标准对数据进行排列

**聚合** 则让我们对数据进行汇总，计算出各种统计量。

| 操作       | 方法                          | 说明                                       | 常用函数/方法示例                                |
|------------|-------------------------------|--------------------------------------------|-------------------------------------------------|
| 排序       | sort_values(by, ascending)    | 根据某列的值进行排序，ascending 控制升降序  | `df.sort_values(by='column')`                      |
| 排序       | sort_index(axis)              | 根据行或列的索引进行排序                   | `df.sort_index(axis=0)`                            |
| 分组聚合   | groupby(by)                   | 按照某列进行分组后，应用聚合函数           | `df.groupby('column') `                            |
| 聚合函数   | agg()                         | 聚合函数，如 sum()、mean()、count() 等     | `df.groupby('column').agg({'value': 'sum'}) `      |
| 多重聚合   | agg([func1, func2])           | 对同一列应用多个聚合函数                   | `df.groupby('column').agg({'value': ['mean', 'sum']})` |
| 分组后排序 | apply(lambda x: x.sort_values(...)) | 在分组后进行排序                         | `df.groupby('column').apply(lambda x: x.sort_values(...))` |
| 透视表     | pivot_table()                 | 创建透视表，根据行、列进行数据汇总         | `df.pivot_table(index='row_col', columns='col_col', values='val_col', aggfunc='sum')` |

### 1. 排序（Sorting）

按列排序 : `df.sort_values(by=['col1'], ascending=True/False)`

按索引排序 : `df.sort_index(axis=0/1)`

In [5]:
import pandas as pd

# 示例数据
data = {'Name': ['Alice', 'Bob', 'Charlie', 'David'],
        'Age': [25, 30, 35, 40],
        'Salary': [50000, 60000, 70000, 80000]}

df = pd.DataFrame(data)
display(df)
# 列降序排序
df_sorted = df.sort_values(by=['Age'], ascending=False)
display(df_sorted)
# 行索引排序
df_sorted_by_index = df_sorted.sort_index(axis=0)  # 行
display(df_sorted_by_index)

,Name,Age,Salary
0,Alice,25,50000
1,Bob,30,60000
2,Charlie,35,70000
3,David,40,80000


,Name,Age,Salary
3,David,40,80000
2,Charlie,35,70000
1,Bob,30,60000
0,Alice,25,50000


,Name,Age,Salary
0,Alice,25,50000
1,Bob,30,60000
2,Charlie,35,70000
3,David,40,80000


### 2. 数据聚合(Aggregation)

**聚合** : 将数据按某些规则进行汇总，通常是对某些列的数据进行求和、求平均数、求最大值、求最小值等操作。Pandas 提供了`groupby()`方法来对数据进行分组，然后应用不同的聚合函数

聚合方法  

`df.groupby(['col1', 'col2']).agg({'col1': ['func1', 'func2'], ...})`：按某些列分组。

聚合函数：如 `sum()`, `mean()`, `count()`, `min()`, `max()`, `std()` 等。 


In [19]:
import pandas as pd

# 示例数据
data = {'Department': ['HR', 'Finance', 'HR', 'IT', 'IT'],
        'Employee': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
        'Salary': [50000, 60000, 55000, 70000, 75000]}

df = pd.DataFrame(data)
display(df)
# 按部门分组，计算各部门的平均薪资
df_grouped = df.groupby('Department')['Salary'].mean()
display(df_grouped)

# 多重聚合函数
# 按部门分组，计算每个部门的薪资的平均值和总和
grouped_multiple = df.groupby('Department').agg({'Salary': ['mean', 'sum']})
print(grouped_multiple)
display(grouped_multiple)

,Department,Employee,Salary
0,HR,Alice,50000
1,Finance,Bob,60000
2,HR,Charlie,55000
3,IT,David,70000
4,IT,Eve,75000


Department
Finance    60000.0
HR         52500.0
IT         72500.0
Name: Salary, dtype: float64

             Salary        
               mean     sum
Department                 
Finance     60000.0   60000
HR          52500.0  105000
IT          72500.0  145000


Salary        
               mean     sum
Department                 
Finance     60000.0   60000
HR          52500.0  105000
IT          72500.0  145000

### 3. 分组后排序

聚合后的数据可以进一步按某列的值进行排序，这样可以找出特定组中 **最重要的值**。

`df.groupby('col1', group_keys=False).apply(lambda x: x.sort.values(by=['col'], ascending=True/False), include_group=False)`

关键说明：

`include_groups=False`（核心参数）：  
告诉 pandas 在 apply 操作时，不要将分组列（这里是 Department）包含在传递给 lambda 函数的子数据框中，避免重复处理，符合 pandas 未来版本的行为规范。

可选：`group_keys=False`（美化输出）：(本人感觉不咋地)  
该参数会在最终结果中隐藏分组键的层级索引，让输出的 DataFrame 结构更简洁（如果需要保留分组键索引，可省略此参数）。

In [15]:
# 按部分分组后，按薪资降序排序
grouped_sorted = df.groupby('Department', group_keys=False).apply(lambda x: x.sort_values(by=['Salary'], ascending=False), include_groups=False)
grouped_sorted2 = df.groupby('Department').apply(lambda x: x.sort_values(by=['Salary'], ascending=False), include_groups=False)
display(grouped_sorted)
display(grouped_sorted2)

,Employee,Salary
1,Bob,60000
2,Charlie,55000
0,Alice,50000
4,Eve,75000
3,David,70000


Employee  Salary
Department                   
Finance    1      Bob   60000
HR         2  Charlie   55000
           0    Alice   50000
IT         4      Eve   75000
           3    David   70000

### 4. 透视表

透视表（Pivot Table）是一个特殊的聚合方式，可以让我们通过行、列和聚合函数对数据进行快速汇总，类似于 Excel 中的透视表

`df.pivot_table(values='col1', index='col2', aggfunc='func')`

In [20]:
# 使用 pivot_table 计算每个部门的薪资平均值
pivot_table = df.pivot_table(values='Salary', index='Department', aggfunc='mean')
display(pivot_table)

,Salary
Department,
Finance,60000.0
HR,52500.0
IT,72500.0
